# Nanobody Design Pipeline – Marmoset SWS1 Opsin (F7GQA6)

End-to-end pipeline:
1. **Setup** – install tools
2. **Target prep** – ECL2 hotspot selection
3. **RFdiffusion** – backbone generation
4. **ProteinMPNN** – sequence design
5. **NanobodyBuilder2** – VHH structure prediction
6. **ColabFold multimer** – complex confidence scoring
7. **PRODIGY** – binding affinity estimate
8. **Filter & rank** – final candidate selection

> **Runtime**: GPU required (A100 recommended). ~3–4 h for 200 designs.


## 0. GPU / Environment Check

In [ ]:
import subprocess, sys
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                    '--format=csv,noheader'], capture_output=True, text=True)
print(r.stdout or 'No GPU detected – switch runtime to GPU!')
print('Python', sys.version)


## 1. Install Dependencies

In [ ]:
# Core bio libraries
!pip install -q biopython numpy pandas matplotlib seaborn pyyaml tqdm

# Sequence scoring
!pip install -q ablang2

# VHH structure prediction
!pip install -q ImmuneBuilder

# PRODIGY (binding affinity)
!pip install -q prodigy-prot

# ColabFold (for multimer scoring — optional, see Section 6)
# !pip install -q 'colabfold[alphafold-without-jax] @ git+https://github.com/sokrypton/ColabFold'


In [ ]:
# Clone RFdiffusion
import os, subprocess, re
if not os.path.exists('/content/RFdiffusion'):
    !git clone -q https://github.com/RosettaCommons/RFdiffusion /content/RFdiffusion

# se3-transformer is bundled in the repo and not available on PyPI
!pip install -q /content/RFdiffusion/env/SE3Transformer

# DGL — detect CUDA version and pick the matching wheel
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
m = re.search(r'CUDA Version: (\d+)', r.stdout)
cuda_major = int(m.group(1)) if m else 0
if cuda_major >= 12:
    !pip install -q dgl -f https://data.dgl.ai/wheels/cu121/repo.html
elif cuda_major == 11:
    !pip install -q dgl -f https://data.dgl.ai/wheels/cu118/repo.html
else:
    !pip install -q dgl  # CPU fallback
import importlib; assert importlib.util.find_spec('dgl'), 'dgl install failed'
print(f'dgl installed (CUDA major={cuda_major})')

# Install RFdiffusion's required dependencies (excluded by --no-deps)
!pip install -q hydra-core omegaconf iopath opt_einsum e3nn pyrsistent decorator

# Install RFdiffusion itself (SE3Transformer + DGL already installed above)
!pip install -q --no-deps -e /content/RFdiffusion

os.environ['RFDIFFUSION_PATH'] = '/content/RFdiffusion'
print('RFdiffusion:', os.environ['RFDIFFUSION_PATH'])


In [ ]:
import os

# Clone ProteinMPNN
if not os.path.exists('/content/ProteinMPNN'):
    !git clone -q https://github.com/dauparas/ProteinMPNN /content/ProteinMPNN
os.environ['PROTEINMPNN_PATH'] = '/content/ProteinMPNN'
print('ProteinMPNN:', os.environ['PROTEINMPNN_PATH'])


## 2. Upload AF2 Structure

Upload the AlphaFold2 model for **F7GQA6** (Marmoset SWS1 Opsin).
You can download it from [AlphaFold DB](https://alphafold.ebi.ac.uk/entry/F7GQA6).


In [ ]:
from google.colab import files
import shutil, os

os.makedirs('data', exist_ok=True)
uploaded = files.upload()          # select F7GQA6_AF2.pdb
for fname in uploaded:
    shutil.move(fname, f'data/{fname}')
    print('Saved:', f'data/{fname}')

PDB_PATH = 'data/F7GQA6_AF2.pdb'  # adjust if filename differs


## 3. Clone Pipeline Repo & Prepare Target

In [ ]:
import os, glob, shutil

BRANCH = 'claude/antibody-design-tool-dF1eH'

if not os.path.exists('/content/Micchan001'):
    !git clone -q -b {BRANCH} https://github.com/Micchan001/Micchan001 /content/Micchan001
else:
    # Ensure we're on the right branch with latest scripts
    !git -C /content/Micchan001 fetch -q origin {BRANCH}
    !git -C /content/Micchan001 checkout -q {BRANCH}
    !git -C /content/Micchan001 pull -q origin {BRANCH}
%cd /content/Micchan001
os.makedirs('data', exist_ok=True)

# Find the uploaded PDB (handles any filename from AlphaFold DB)
candidates = glob.glob('/content/data/*.pdb') + glob.glob('/content/*.pdb')
if not candidates:
    raise FileNotFoundError('PDB not found. Please run Section 2 (Upload AF2 Structure) first.')
src_pdb = candidates[0]
dst_pdb = 'data/F7GQA6_AF2.pdb'
if not os.path.exists(dst_pdb):
    shutil.copy(src_pdb, dst_pdb)
    print(f'Copied {src_pdb} → {dst_pdb}')
else:
    print(f'Already exists: {dst_pdb}')


In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'scripts/01_prepare_target.py',
     '--config', 'configs/design_config.yaml',
     '--pdb', 'data/F7GQA6_AF2.pdb'],
    capture_output=True,
    text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:\n', result.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f'01_prepare_target.py failed (exit code {result.returncode}).\n'
        'See output above for details.'
    )


In [ ]:
import json, os
import pandas as pd

# --- hotspots ---
if os.path.exists('data/F7GQA6_hotspots.json'):
    with open('data/F7GQA6_hotspots.json') as f:
        hotspots = json.load(f)
    print('Hotspot residues:', hotspots['hotspot_residues'])
else:
    # Fallback: use candidates from config (skip pLDDT/SASA filtering)
    import yaml
    cfg = yaml.safe_load(open('configs/design_config.yaml'))
    fallback_ids = cfg['epitope']['hotspot_candidates'][:6]
    chain = cfg['target']['chain_id']
    hotspots = {'hotspot_residues': [f'{chain}{r}' for r in fallback_ids]}
    with open('data/F7GQA6_hotspots.json', 'w') as f:
        json.dump(hotspots, f, indent=2)
    # also write fallback topology if missing
    print('[WARN] 01_prepare_target.py did not produce output; using config defaults.')
    print('Hotspot residues (fallback):', hotspots['hotspot_residues'])
    # update config
    cfg['rfdiffusion']['hotspot_residues'] = hotspots['hotspot_residues']
    with open('configs/design_config.yaml', 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

# --- topology table (optional) ---
if os.path.exists('data/F7GQA6_topology.tsv'):
    topo = pd.read_csv('data/F7GQA6_topology.tsv', sep='\t')
    print('\nECL2 residues:')
    print(topo[topo['region']=='ECL2'][['resid','resname','plddt','sasa']].to_string(index=False))
else:
    print('[INFO] Topology table not found (skipping display).')


## 4. RFdiffusion – Backbone Generation

Generates 200 de-novo nanobody backbones targeting ECL2.
~2–3 h on A100. Reduce `NUM_DESIGNS` for a quick test.


In [ ]:
NUM_DESIGNS = 200   # set to 10 for a quick test

import yaml
with open('configs/design_config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['rfdiffusion']['num_designs'] = NUM_DESIGNS
with open('configs/design_config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
print(f'Set num_designs = {NUM_DESIGNS}')


In [ ]:
import os, importlib, subprocess, sys, re
os.environ.setdefault('RFDIFFUSION_PATH', '/content/RFdiffusion')
os.environ.setdefault('PROTEINMPNN_PATH', '/content/ProteinMPNN')

# Install dgl here if Cell 5 was skipped or failed
if not importlib.util.find_spec('dgl'):
    r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    m = re.search(r'CUDA Version: (\d+)', r.stdout)
    cuda = int(m.group(1)) if m else 0
    url = ('https://data.dgl.ai/wheels/cu121/repo.html' if cuda >= 12
           else 'https://data.dgl.ai/wheels/cu118/repo.html' if cuda == 11
           else None)
    cmd = [sys.executable, '-m', 'pip', 'install', 'dgl']
    if url: cmd += ['-f', url]
    subprocess.run(cmd, check=True)
    print('dgl installed')

!python scripts/02_run_rfdiffusion.py \
    --config configs/design_config.yaml \
    --output-dir data/rfdiffusion_outputs \
    --run


In [ ]:
# Count outputs
from pathlib import Path
pdbs = list(Path('data/rfdiffusion_outputs').glob('design_*.pdb'))
print(f'Generated {len(pdbs)} backbone PDBs')


## 5. ProteinMPNN – Sequence Design

Designs 8 sequences per backbone (CDR1/2/3 redesigned; framework fixed).


In [ ]:
import os, importlib, subprocess, sys, re
os.environ.setdefault('RFDIFFUSION_PATH', '/content/RFdiffusion')
os.environ.setdefault('PROTEINMPNN_PATH', '/content/ProteinMPNN')

# Install dgl here if Cell 5 was skipped or failed
if not importlib.util.find_spec('dgl'):
    r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    m = re.search(r'CUDA Version: (\d+)', r.stdout)
    cuda = int(m.group(1)) if m else 0
    url = ('https://data.dgl.ai/wheels/cu121/repo.html' if cuda >= 12
           else 'https://data.dgl.ai/wheels/cu118/repo.html' if cuda == 11
           else None)
    cmd = [sys.executable, '-m', 'pip', 'install', 'dgl']
    if url: cmd += ['-f', url]
    subprocess.run(cmd, check=True)
    print('dgl installed')

!python scripts/02_run_rfdiffusion.py \
    --config configs/design_config.yaml \
    --output-dir data/rfdiffusion_outputs \
    --run --mpnn


In [ ]:
fastas = list(Path('data/rfdiffusion_outputs/mpnn_seqs').glob('**/*.fa'))
print(f'Sequence FASTA files: {len(fastas)}')
if fastas:
    print(open(fastas[0]).read()[:500])


## 6. NanobodyBuilder2 – VHH Structure Prediction

Predicts 3D structure of each VHH sequence (fast, ~1 s/seq).


In [ ]:
from ImmuneBuilder import NanobodyBuilder
import os
from pathlib import Path

builder = NanobodyBuilder()
os.makedirs('data/nb_structures', exist_ok=True)

# Parse all FASTA sequences from ProteinMPNN output
def parse_fasta(path):
    seqs = {}
    name, buf = None, []
    for line in open(path):
        line = line.strip()
        if line.startswith('>'):
            if name: seqs[name] = ''.join(buf)
            name, buf = line[1:].split()[0], []
        else:
            buf.append(line)
    if name: seqs[name] = ''.join(buf)
    return seqs

all_seqs = {}
for fa in Path('data/rfdiffusion_outputs/mpnn_seqs').glob('**/*.fa'):
    all_seqs.update(parse_fasta(str(fa)))

print(f'Total sequences: {len(all_seqs)}')

failed = []
for name, seq in list(all_seqs.items())[:200]:   # cap at 200
    out = f'data/nb_structures/{name}.pdb'
    if os.path.exists(out):
        continue
    try:
        nb = builder.predict({'H': seq})
        nb.save(out)
    except Exception as e:
        failed.append((name, str(e)))

predicted = list(Path('data/nb_structures').glob('*.pdb'))
print(f'Predicted: {len(predicted)}, Failed: {len(failed)}')


## 7. ColabFold Multimer – Complex Confidence Scoring

Scores nanobody–opsin complex using AF2-multimer.
Generates ipTM, pTM, and interface PAE for each candidate.


In [ ]:
# Build FASTA for ColabFold multimer (opsin:nanobody)
from Bio.PDB import PDBParser

def pdb_to_seq(pdb_path, chain_id='A'):
    aa = {'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q',
          'GLU':'E','GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K',
          'MET':'M','PHE':'F','PRO':'P','SER':'S','THR':'T','TRP':'W',
          'TYR':'Y','VAL':'V'}
    p = PDBParser(QUIET=True)
    s = p.get_structure('x', pdb_path)
    return ''.join(aa.get(r.get_resname(),'X')
                   for r in s[0][chain_id].get_residues()
                   if r.get_id()[0]==' ')

opsin_seq = pdb_to_seq('data/F7GQA6_AF2.pdb', 'A')

os.makedirs('data/colabfold_inputs', exist_ok=True)
nb_pdbs = sorted(Path('data/nb_structures').glob('*.pdb'))[:50]  # top 50

fasta_out = 'data/colabfold_inputs/complexes.fasta'
with open(fasta_out, 'w') as f:
    for nb_pdb in nb_pdbs:
        nb_seq = pdb_to_seq(str(nb_pdb), 'H')
        name = nb_pdb.stem
        f.write(f'>{name}\n{opsin_seq}:{nb_seq}\n')

print(f'Written {len(nb_pdbs)} complex FASTAs to {fasta_out}')


In [ ]:
# Run ColabFold (requires colabfold installation; ~2 min/complex on A100)
# Uncomment to run:
# !colabfold_batch data/colabfold_inputs/complexes.fasta \
#     data/colabfold_outputs \
#     --model-type alphafold2_multimer_v3 \
#     --num-recycle 3 \
#     --num-models 1

print('ColabFold block ready. Uncomment the command above to run.')


## 8. PRODIGY – Binding Affinity Estimation

Estimates ΔG (kcal/mol) and Kd for each complex structure.


In [ ]:
import subprocess, os
from pathlib import Path

def run_prodigy(pdb_path, chain_a='A', chain_b='B'):
    r = subprocess.run(
        ['prodigy', pdb_path, '--selection', chain_a, chain_b,
         '--temperature', '25'],
        capture_output=True, text=True, timeout=30
    )
    dG = Kd = None
    for line in r.stdout.splitlines():
        if 'Predicted binding affinity' in line:
            dG = float(line.split(':')[-1].split()[0])
        if 'Predicted dissociation constant' in line:
            Kd = line.split(':')[-1].strip()
    return dG, Kd

# RFdiffusion outputs (design_*.pdb) contain both chains A (target) and B (nanobody)
complex_pdbs = sorted(Path('data/rfdiffusion_outputs').glob('design_*.pdb'))
if complex_pdbs:
    dG, Kd = run_prodigy(str(complex_pdbs[0]))
    print(f'Test: {complex_pdbs[0].name}  ΔG={dG} kcal/mol  Kd={Kd}')
else:
    print('No RFdiffusion PDBs found. Run Section 4 first.')


## 9. Filter & Rank Final Candidates

In [ ]:
!python scripts/03_filter_designs.py \
    --designs data/rfdiffusion_outputs \
    --config configs/design_config.yaml \
    --output data/filtered_candidates \
    --top-n 10


In [ ]:
import pandas as pd

top = pd.read_csv('data/filtered_candidates/top_candidates.tsv', sep='\t')
display_cols = [c for c in ['name','composite_score','iptm','pae_interface',
                             'ablang_score','prodigy_dG','interface_nb']
                if c in top.columns]
print(top[display_cols].to_string(index=False))


## 10. Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

all_df = pd.read_csv('data/filtered_candidates/all_scores.tsv', sep='\t')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Composite score distribution
axes[0].hist(all_df['composite_score'].dropna(), bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Composite Score Distribution')
axes[0].set_xlabel('Composite Score')

# ipTM vs PAE interface
if 'iptm' in all_df and 'pae_interface' in all_df:
    sc = axes[1].scatter(all_df['pae_interface'], all_df['iptm'],
                         c=all_df['composite_score'], cmap='viridis', alpha=0.7)
    axes[1].set_xlabel('PAE Interface (Å)')
    axes[1].set_ylabel('ipTM')
    axes[1].set_title('ipTM vs PAE Interface')
    plt.colorbar(sc, ax=axes[1], label='Composite Score')
else:
    axes[1].text(0.5, 0.5, 'ColabFold scores\nnot available', ha='center', va='center')
    axes[1].set_title('ipTM vs PAE Interface')

# PRODIGY ΔG
if 'prodigy_dG' in all_df:
    axes[2].hist(all_df['prodigy_dG'].dropna(), bins=30, color='coral', edgecolor='white')
    axes[2].axvline(-8, color='red', linestyle='--', label='threshold (-8)')
    axes[2].set_title('PRODIGY ΔG Distribution')
    axes[2].set_xlabel('ΔG (kcal/mol)')
    axes[2].legend()
else:
    axes[2].text(0.5, 0.5, 'PRODIGY scores\nnot available', ha='center', va='center')
    axes[2].set_title('PRODIGY ΔG Distribution')

plt.tight_layout()
plt.savefig('data/filtered_candidates/score_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: data/filtered_candidates/score_overview.png')


## 11. Download Results

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os


zip_path = 'nanobody_candidates.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in Path('data/filtered_candidates').rglob('*'):
        if f.is_file():
            zf.write(f, f.relative_to('data'))

print(f'Archive: {zip_path} ({os.path.getsize(zip_path)/1024:.0f} KB)')
files.download(zip_path)
